In [0]:
-- Daily spend per advertiser with a 7-day rolling average, and each day's % change vs the prior day.

with daly_spend as(
    select event_date,advertiser_id, advertiser_name, total_spend_usd from gold.daily_advertiser_spend
)

  --7-day rolling average: window of the CURRENT row and 6 preceding
  -- days, partitioned per advertiser so one advertiser's spend never
  -- blends into another's average

  -- % change vs prior day: LAG grabs yesterday's spend for the same
  -- advertiser; NULLIF guards the divide-by-zero case where yesterday's
  -- spend was exactly 0 (would otherwise throw or return infinity)

select 
event_date, 
advertiser_id, 
advertiser_name, total_spend_usd, avg(total_spend_usd) over (partition by advertiser_id order by event_date rows between 6 preceding and current row) as rolling_avg, round(((total_spend_usd - lag(total_spend_usd) over (partition by advertiser_id order by event_date))/nullif(lag(total_spend_usd) over (partition by advertiser_id order by event_date),0))*100,2) as pct_change
from daly_spend
order by advertiser_id, event_date;



-- we can check the consecutive days for a specific advertiser

select * from(WITH daily_spend AS (
  SELECT
    event_date,
    advertiser_id,
    advertiser_name,
    total_spend_usd
  FROM gold.daily_advertiser_spend
)

SELECT
  event_date,
  advertiser_id,
  advertiser_name,
  total_spend_usd,

  -- 7-day rolling average: window of the CURRENT row and 6 preceding
  -- days, partitioned per advertiser so one advertiser's spend never
  -- blends into another's average
  AVG(total_spend_usd) OVER (
    PARTITION BY advertiser_id
    ORDER BY event_date
    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
  ) AS rolling_7day_avg_spend,

  -- % change vs prior day: LAG grabs yesterday's spend for the same
  -- advertiser; NULLIF guards the divide-by-zero case where yesterday's
  -- spend was exactly 0 (would otherwise throw or return infinity)
  ROUND(
    (total_spend_usd - LAG(total_spend_usd) OVER (
        PARTITION BY advertiser_id ORDER BY event_date
     )) 
    / NULLIF(LAG(total_spend_usd) OVER (
        PARTITION BY advertiser_id ORDER BY event_date
      ), 0) * 100,
    2
  ) AS pct_change_vs_prior_day

FROM daily_spend
ORDER BY advertiser_id, event_date
)
where advertiser_id ='adv_1334'
order by event_date
limit 10;

-- Q2 — Longest streak of consecutive active days per advertiser 

WITH active_days AS (
  -- one row per advertiser per day they had at least one event —
  -- distinct is essential here, we only care about "active or not,"
  -- not how many events that day
  SELECT DISTINCT
    advertiser_id,
    event_date
  FROM gold.daily_advertiser_spend
  WHERE event_count >= 1
),
islands AS (
  SELECT
    advertiser_id,
    event_date,
    -- date_diff between the actual date and a sequential row number
    -- (converted to a date offset) is constant across any run of
    -- consecutive calendar dates — that constant is the "island ID"
    DATE_SUB(
      event_date,
      ROW_NUMBER() OVER (PARTITION BY advertiser_id ORDER BY event_date)
    ) AS island_marker
  FROM active_days
), streaks AS (
  SELECT
    advertiser_id,
    island_marker,
    COUNT(*) AS streak_length,
    MIN(event_date) AS streak_start,
    MAX(event_date) AS streak_end
  FROM islands
  GROUP BY advertiser_id, island_marker
)

SELECT
  advertiser_id,
  streak_length AS longest_streak_days,
  streak_start,
  streak_end
FROM streaks
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY advertiser_id ORDER BY streak_length DESC, streak_start ASC
) = 1
ORDER BY advertiser_id;

